# A1b: Attacker Optimal Trajectory — Bellman / Direct Collocation
## CasADi + IPOPT

**Problem:** Given fixed sensor placement `z_d`, find the attacker's optimal trajectory minimizing:

    J_A = alpha_1 * xi(T) + alpha_2 * T_norm

where `xi(T) = integral_0^T sum_k lambda_k dt` is the total hazard integral
(log-sum-exp formulation — monotone with J_PD, avoids exp nonlinearity in running cost).

**Method:** Direct collocation (Hermite-Simpson) with CasADi + IPOPT.

**Scope:** Attacker side only. Defender placement `z_d` is fixed heuristically.
Stackelberg outer loop (sweep over `z_d`) will be added separately.

**Formulation reference:** A1b Problem Formulation, Sections 6-9.

In [ ]:
# pip install casadi
import casadi as ca
import numpy as np
import matplotlib.pyplot as plt

print(f'CasADi version: {ca.__version__}')
print('Imports OK')

---
## Section 1 — Constants
All tunable parameters in one place. Change these freely.

In [ ]:
# ── Corridor ──────────────────────────────────────────────────────────────────
Z_START = 0.0
Z_GOAL  = 20000.0    # m  — 20 km corridor gives meaningful glide decisions

# ── Defender placement (heuristic — Stackelberg loop added later) ─────────────
Z_D = 6000.0         # m  — sensor at 30% of corridor

# ── Airframe (parabolic drag polar) ───────────────────────────────────────────
C_D0      = 0.025    # zero-lift drag coefficient
K_INDUCED = 0.045    # induced drag factor k = 1/(pi*e*AR)
W         = 150.0    # N  vehicle weight
RHO       = 1.225    # kg/m^3  sea-level air density
S_WING    = 0.8      # m^2  wing reference area

# ── Powered phase ─────────────────────────────────────────────────────────────
V_CLIMB   = 15.0     # m/s  horizontal speed during climb (constant)
H_DOT_MAX = 8.0      # m/s  maximum climb rate

# ── Glide phase constraints ────────────────────────────────────────────────────
GAMMA_MIN_DEG = -45.0
GAMMA_MAX_DEG = -2.0
GAMMA_MIN     = np.radians(GAMMA_MIN_DEG)
GAMMA_MAX     = np.radians(GAMMA_MAX_DEG)

# ── Direct collocation ─────────────────────────────────────────────────────────
N_COLL = 80          # collocation intervals — more = smoother, slower

# ── Cost weights ──────────────────────────────────────────────────────────────
ALPHA_1 = 0.7        # weight on xi (detection)
ALPHA_2 = 0.3        # weight on T_norm (flight time)

# ── Sensor parameters ─────────────────────────────────────────────────────────
N_ACOUSTIC    = 6.0  # propeller dipole exponent (Lighthill 1952, Curle 1955)
SIGMA_PSI_DEG = 30.0 # RCS Gaussian spread (deg) (Inanc et al. 2008 JGCD)
FOV_HALF_DEG  = 45.0 # camera FOV half-angle (deg)
CAM_ELEV_DEG  = 45.0 # camera boresight elevation (deg)

# Kappas — calibrated in Section 3
KAPPA_AC   = 1.0
KAPPA_DOPP = 1.0
KAPPA_RCS  = 1.0
KAPPA_CAM  = 1.0

print(f'Constants set.')
print(f'  Corridor: z=[{Z_START:.0f}, {Z_GOAL:.0f}] m')
print(f'  Sensor:   z_d={Z_D:.0f} m  ({100*Z_D/Z_GOAL:.0f}% of corridor)')
print(f'  N_COLL:   {N_COLL}')

---
## Section 2 — Glide Polar

Parabolic drag polar enforces the **nonholonomic constraint**:
once `gamma` is chosen, trim airspeed `v(gamma)` is determined.
The attacker cannot choose `v` and `gamma` independently.

We need two versions:
- **NumPy** — for calibration, cost evaluation, and plotting
- **CasADi symbolic** — for the NLP (automatic differentiation for IPOPT)

In [ ]:
# ── Best-glide quantities ──────────────────────────────────────────────────────
CL_STAR    = np.sqrt(C_D0 / K_INDUCED)           # C_L at max L/D
CD_STAR    = C_D0 + K_INDUCED * CL_STAR**2       # = 2*C_D0 at best glide
LD_MAX     = CL_STAR / CD_STAR                    # (L/D)_max
GAMMA_STAR = -np.arctan(1.0 / LD_MAX)             # best glide angle (negative)
V_STAR     = np.sqrt(2*W*np.cos(abs(GAMMA_STAR)) / (RHO*S_WING*CL_STAR))
T_MAX      = (Z_GOAL - Z_START) / V_STAR          # normalization for T_norm

print(f'Glide polar:')
print(f'  (L/D)_max  = {LD_MAX:.2f}')
print(f'  gamma*     = {np.degrees(GAMMA_STAR):.2f} deg  (best glide angle)')
print(f'  v*         = {V_STAR:.2f} m/s  (best glide speed)')
print(f'  T_max      = {T_MAX:.0f} s')
print(f'  CD* = 2*C_D0? {np.isclose(CD_STAR, 2*C_D0)} (sanity check)')


# ── NumPy polar (calibration + plotting) ──────────────────────────────────────
def glide_speed_np(gamma):
    """
    Trim airspeed at glide angle gamma — NumPy version.
    Inverts the force balance: tan|gamma| = (C_D0 + k*C_L^2) / C_L
    Takes lower root (stable high-speed branch).
    Stable branch: steeper gamma -> lower C_L -> higher v.
    """
    gamma = float(np.clip(gamma, GAMMA_MIN, GAMMA_MAX))
    tan_g = np.tan(abs(gamma))
    disc  = tan_g**2 - 4.0*K_INDUCED*C_D0
    CL    = CL_STAR if disc < 0 else max(
                (tan_g - np.sqrt(disc)) / (2.0*K_INDUCED), 0.01)
    return float(np.sqrt(2*W*np.cos(abs(gamma)) / (RHO*S_WING*CL)))


# ── CasADi symbolic polar (NLP) ───────────────────────────────────────────────
def glide_speed_ca(gamma_sym):
    """
    Trim airspeed — CasADi symbolic version.
    Must use ca.* functions throughout for automatic differentiation.
    disc is clamped to avoid sqrt(negative) when gamma shallower than gamma*.
    """
    tan_g    = ca.tan(ca.fabs(gamma_sym))
    disc     = ca.fmax(tan_g**2 - 4.0*K_INDUCED*C_D0, 1e-8)
    CL       = ca.fmax((tan_g - ca.sqrt(disc)) / (2.0*K_INDUCED), 0.01)
    return ca.sqrt(2*W*ca.cos(ca.fabs(gamma_sym)) / (RHO*S_WING*CL))


# Sanity checks
assert abs(glide_speed_np(GAMMA_STAR) - V_STAR) < 1.0, 'polar sanity failed'
assert glide_speed_np(np.radians(-30)) > glide_speed_np(np.radians(-10)),     'stable branch: steeper should be faster'
print(f'Polar sanity checks passed.')
print(f'  glide_speed(gamma*) = {glide_speed_np(GAMMA_STAR):.2f} m/s = v* ✓')

---
## Section 3 — Sensor Models

Four sensors collocated at `(Z_D, 0)`.
Same two-version pattern: NumPy for calibration, CasADi for NLP.

**Poisson fusion** (Zabarankin et al. 2006):
`J_PD = 1 - exp(-xi(T))`  where  `xi(T) = integral_0^T sum_k lambda_k dt`

**Log-sum-exp formulation:** we minimize `xi(T)` directly.
This is equivalent to minimizing `J_PD` (monotone relationship)
and avoids the `exp` nonlinearity in the running cost.

In [ ]:
# Camera boresight unit vector (precomputed once)
_elev = np.radians(CAM_ELEV_DEG)
B_HAT = np.array([-np.cos(_elev), np.sin(_elev)])


# ── NumPy geometry helpers ────────────────────────────────────────────────────
def sensor_range_np(z, h, z_d=None):
    if z_d is None: z_d = Z_D
    return float(np.sqrt((z_d-z)**2 + h**2))

def los_angle_np(z, h, z_d=None):
    if z_d is None: z_d = Z_D
    dz = abs(z_d - z)
    return float(np.pi/2 if dz < 1e-6 else np.arctan2(h, dz))

def radial_vel_np(z, h, v, gamma, z_d=None):
    if z_d is None: z_d = Z_D
    return float(v * np.cos(los_angle_np(z, h, z_d) - abs(gamma)))

def aspect_deg_np(z, h, gamma, z_d=None):
    if z_d is None: z_d = Z_D
    psi = abs((np.pi/2 + gamma) - los_angle_np(z, h, z_d))
    return float(np.degrees(psi))

def boresight_np(z, h, z_d=None):
    if z_d is None: z_d = Z_D
    u = np.array([z-z_d, h])
    n = np.linalg.norm(u)
    if n < 1e-6: return 0.0
    return float(np.arccos(np.clip(np.dot(B_HAT, u/n), -1.0, 1.0)))


# ── NumPy sensor hazard rates ─────────────────────────────────────────────────
def lam_acoustic_np(z, h, v, gamma, phase, z_d=None):
    """Phase 1 only. lambda = kappa * v^n / r^2. Source: Lighthill 1952."""
    if phase != 1: return 0.0
    r = sensor_range_np(z, h, z_d)
    if r < 1.0: return 0.0
    return KAPPA_AC * (v**N_ACOUSTIC) / (r**2)

def lam_doppler_np(z, h, v, gamma, phase, z_d=None):
    """Both phases. lambda = kappa * v_r^2 / r^4. Source: Skolnik 2008."""
    r = sensor_range_np(z, h, z_d)
    if r < 1.0: return 0.0
    vr = radial_vel_np(z, h, v, gamma, z_d)
    return KAPPA_DOPP * (vr**2) / (r**4)

def lam_rcs_np(z, h, v, gamma, phase, z_d=None):
    """Both phases. Gaussian RCS model. Source: Inanc et al. 2008."""
    r = sensor_range_np(z, h, z_d)
    if r < 1.0: return 0.0
    sig_r = np.radians(SIGMA_PSI_DEG)
    psi_r = np.radians(aspect_deg_np(z, h, gamma, z_d))
    sigma = np.exp(-((psi_r - np.pi/2)**2) / (2*sig_r**2))
    return KAPPA_RCS * sigma / (r**4)

def lam_camera_np(z, h, v, gamma, phase, z_d=None):
    """Both phases, within FOV. lambda = kappa * cos^2(phi) / r^2."""
    r = sensor_range_np(z, h, z_d)
    if r < 1.0: return 0.0
    phi = boresight_np(z, h, z_d)
    if phi > np.radians(FOV_HALF_DEG): return 0.0
    return KAPPA_CAM * (np.cos(phi)**2) / (r**2)

def lam_total_np(z, h, v, gamma, phase, z_d=None):
    return (lam_acoustic_np(z, h, v, gamma, phase, z_d) +
            lam_doppler_np( z, h, v, gamma, phase, z_d) +
            lam_rcs_np(     z, h, v, gamma, phase, z_d) +
            lam_camera_np(  z, h, v, gamma, phase, z_d))


# ── CasADi sensor (symbolic, glide phase only) ────────────────────────────────
# The NLP covers Phase 2 only — acoustic excluded (engine off).
# Hard FOV cutoff replaced with smooth sigmoid gate (non-differentiable otherwise).
def lam_total_ca(z_s, h_s, gamma_s, z_d=None):
    """
    Symbolic total hazard rate for glide phase (phase=2).
    Acoustic excluded. Camera FOV uses smooth sigmoid gate.
    """
    if z_d is None: z_d = Z_D
    z_d_f = float(z_d)

    # Range
    r2    = (z_d_f - z_s)**2 + h_s**2
    r     = ca.sqrt(ca.fmax(r2, 1.0))   # clamp to avoid r=0

    # LOS angle
    dz      = ca.fabs(z_d_f - z_s)
    theta   = ca.atan2(h_s, ca.fmax(dz, 1e-6))

    # Trim airspeed (nonholonomic constraint)
    v = glide_speed_ca(gamma_s)

    # Doppler: lambda = kappa * v_r^2 / r^4
    vr       = v * ca.cos(theta - ca.fabs(gamma_s))
    lam_dopp = KAPPA_DOPP * (vr**2) / (r**4)

    # RCS: lambda = kappa * sigma(psi) / r^4
    ventral  = ca.pi/2.0 + gamma_s
    psi      = ca.fabs(ventral - theta)
    sig_r    = float(np.radians(SIGMA_PSI_DEG))
    sigma    = ca.exp(-((psi - ca.pi/2)**2) / (2*sig_r**2))
    lam_rcs  = KAPPA_RCS * sigma / (r**4)

    # Camera: lambda = kappa * cos^2(phi) / r^2 * gate(phi)
    bx    = float(-np.cos(_elev))
    by    = float(np.sin(_elev))
    u_n   = ca.sqrt(ca.fmax((z_s-z_d_f)**2 + h_s**2, 1e-6))
    dot   = ca.fmin(ca.fmax(bx*(z_s-z_d_f)/u_n + by*h_s/u_n, -1.0), 1.0)
    phi   = ca.acos(dot)
    fov_r = float(np.radians(FOV_HALF_DEG))
    # Smooth sigmoid gate: ~1 inside FOV, ~0 outside
    # k=50 gives sharp transition (~2 deg transition width)
    gate  = 1.0 / (1.0 + ca.exp(50.0*(phi - fov_r)))
    lam_cam = KAPPA_CAM * (ca.cos(phi)**2) / (r**2) * gate

    return lam_dopp + lam_rcs + lam_cam


print('Sensor models defined.')
print('Sanity checks:')
print(f'  acoustic(phase=2) = {lam_acoustic_np(3000, 500, 20, -0.15, 2):.1f}  (should be 0)')
print(f'  acoustic(phase=1) = {lam_acoustic_np(3000, 500, 20, -0.15, 1):.4e}  (should be >0)')
print(f'  doppler           = {lam_doppler_np( 3000, 500, 20, -0.15, 2):.4e}')
print(f'  rcs               = {lam_rcs_np(     3000, 500, 20, -0.15, 2):.4e}')
print(f'  camera            = {lam_camera_np(  3000, 500, 20, -0.15, 2):.4e}')

---
## Section 4 — Trajectory Simulation and Cost Functions

`simulate()` is used for:
1. Sensor calibration (Section 5)
2. Verifying the NLP solution after IPOPT solves
3. Comparing optimal vs straight-line trajectories

**Key design:** `z_sw` is the decision variable. `h_sw` is derived:
`h_sw = h_dot * z_sw / V_CLIMB`

**Cost formulation (log-sum-exp):**
`J_A = alpha_1 * xi(T) + alpha_2 * T_norm`
where minimizing `xi` is equivalent to minimizing `J_PD` (monotone).

In [ ]:
def simulate(h_dot, z_sw, gammas_seg, z_d=None, dt=0.5):
    """
    Simulate full hybrid trajectory.

    Phase 1: powered climb (z_start, 0) -> (z_sw, h_sw)
    Phase 2: segmented glide (z_sw, h_sw) -> (z_goal, 0)

    Parameters
    ----------
    h_dot      : float   climb rate (m/s), in [0.1, H_DOT_MAX]
    z_sw       : float   engine cutoff horizontal position (m)
                         h_sw is DERIVED: h_sw = h_dot * z_sw / V_CLIMB
    gammas_seg : array   N glide angles, one per altitude segment
    z_d        : float   sensor position (defaults to Z_D)

    Returns
    -------
    dict: t, z, h, v, gamma, phase arrays + h_sw, z_sw, delta scalars
    """
    if z_d is None: z_d = Z_D
    h_dot = float(np.clip(h_dot, 0.1, H_DOT_MAX))
    z_sw  = float(z_sw)
    N     = len(gammas_seg)

    # h_sw derived — NOT a free variable
    t_climb = z_sw / V_CLIMB
    h_sw    = h_dot * t_climb
    delta   = z_sw + h_sw * LD_MAX - Z_GOAL

    if delta < 0:
        raise ValueError(f'Infeasible: delta={delta:.1f}m. '
                         f'z_sw={z_sw:.0f}m, h_sw={h_sw:.0f}m')
    if z_sw >= Z_GOAL:
        raise ValueError(f'z_sw={z_sw:.1f} >= Z_GOAL={Z_GOAL:.0f}')

    gamma_climb = np.arctan2(h_dot, V_CLIMB)
    t_arr, z_arr, h_arr, v_arr, g_arr, ph_arr = [], [], [], [], [], []

    # ── Phase 1: powered climb ────────────────────────────────────────────────
    t, z, h = 0.0, Z_START, 0.0
    while h < h_sw - 1e-3 and z < z_sw - 1e-3:
        t_arr.append(t); z_arr.append(z); h_arr.append(h)
        v_arr.append(V_CLIMB); g_arr.append(gamma_climb); ph_arr.append(1)
        z += V_CLIMB * dt
        h += h_dot   * dt
        t += dt

    # Exact switch point
    t_arr.append(t_climb); z_arr.append(z_sw); h_arr.append(h_sw)
    v_arr.append(V_CLIMB); g_arr.append(gamma_climb); ph_arr.append(1)

    # ── Phase 2: segmented glide ──────────────────────────────────────────────
    # N altitude bands from h_sw down to 0, each with its own gamma
    bounds    = np.linspace(h_sw, 0.0, N + 1)
    t         = t_climb
    z         = z_sw
    h         = h_sw
    delta_rem = delta
    gamma     = GAMMA_STAR
    v         = glide_speed_np(gamma)

    while h > 1e-3 and z < Z_GOAL:
        # Current altitude segment
        seg_idx = N - 1
        for i in range(N):
            if h >= bounds[i + 1]:
                seg_idx = i
                break

        gamma_desired = float(np.clip(gammas_seg[seg_idx], GAMMA_MIN, GAMMA_MAX))

        # Dynamic gamma ceiling: must be steep enough to reach ground at Z_GOAL
        z_rem = Z_GOAL - z
        gamma_ceiling = (min(GAMMA_MAX, -np.arctan(h / z_rem))
                         if z_rem > 1.0 and h > 0.1 else GAMMA_MIN)

        # Budget enforcement: force gamma* if Delta exhausted
        if delta_rem <= 0.0:
            delta_rem     = 0.0
            gamma_desired = GAMMA_STAR

        gamma   = float(np.clip(gamma_desired, GAMMA_MIN, gamma_ceiling))
        v       = glide_speed_np(gamma)
        z_dot   = v * np.cos(abs(gamma))
        h_dot_g = v * np.sin(gamma)   # negative

        t_arr.append(t); z_arr.append(z); h_arr.append(h)
        v_arr.append(v); g_arr.append(gamma); ph_arr.append(2)

        # Range budget decrement
        tan_g     = np.tan(abs(gamma))
        cot_gamma = 1.0 / tan_g if tan_g > 1e-6 else 1e6
        delta_rem = max(delta_rem - (LD_MAX - cot_gamma)*abs(h_dot_g)*dt, 0.0)

        z += z_dot  * dt
        h  = max(h + h_dot_g * dt, 0.0)
        t += dt

    # Final point at ground
    t_arr.append(t); z_arr.append(min(z, Z_GOAL)); h_arr.append(0.0)
    v_arr.append(v); g_arr.append(gamma); ph_arr.append(2)

    return {
        't':     np.array(t_arr),
        'z':     np.array(z_arr),
        'h':     np.array(h_arr),
        'v':     np.array(v_arr),
        'gamma': np.array(g_arr),
        'phase': np.array(ph_arr),
        'h_sw':  h_sw,
        'z_sw':  z_sw,
        'delta': delta,
    }


def compute_costs(traj, z_d=None):
    """
    Compute J_A, J_PD, xi, T_norm and running J_PD array.

    Uses log-sum-exp formulation:
        J_A = alpha_1 * xi(T) + alpha_2 * T_norm
        J_PD = 1 - exp(-xi)   [for reporting only, not optimized directly]
    """
    if z_d is None: z_d = Z_D
    n   = len(traj['t'])
    lam = np.array([lam_total_np(traj['z'][i], traj['h'][i], traj['v'][i],
                                  traj['gamma'][i], traj['phase'][i], z_d)
                    for i in range(n)])
    dt_arr  = np.diff(traj['t'])
    cumint  = np.concatenate([[0], np.cumsum(0.5*(lam[:-1]+lam[1:])*dt_arr)])
    xi      = float(cumint[-1])
    jpd_run = 1.0 - np.exp(-cumint)
    J_PD    = float(jpd_run[-1])
    T       = float(traj['t'][-1] - traj['t'][0])
    T_norm  = float(np.clip(T / T_MAX, 0.0, 1.0))
    J_A     = ALPHA_1 * xi + ALPHA_2 * T_norm
    return {'J_A': J_A, 'J_PD': J_PD, 'xi': xi, 'T_norm': T_norm,
            'jpd_run': jpd_run, 'lam': lam}


# Quick test
_z_sw_test = Z_GOAL / (1 + H_DOT_MAX*LD_MAX/V_CLIMB) * 1.3
_traj_test = simulate(H_DOT_MAX, _z_sw_test, np.full(10, GAMMA_STAR))
_costs     = compute_costs(_traj_test)
print(f'Simulation test: z_sw={_traj_test["z_sw"]:.0f}m, h_sw={_traj_test["h_sw"]:.0f}m')
print(f'Costs test:      xi={_costs["xi"]:.4f}, J_PD={_costs["J_PD"]:.4f}')

---
## Section 5 — Sensor Calibration

**Goal:** Each sensor contributes `TARGET_PER_SENSOR` to the total hazard integral
on a representative reference trajectory. Total `xi ~ 4 * TARGET`, giving a
meaningful `J_PD` — not saturated at 1.0, not negligible.

**Method:** Simulate reference trajectory with `kappa=1`, measure raw integral
per sensor, scale each kappa so its integral equals the target.

In [ ]:
TARGET_PER_SENSOR = 0.15   # each sensor contributes 0.15
                            # total xi ~ 0.60 -> J_PD ~ 0.45

# Reference trajectory: full power climb, engine cut just before sensor,
# straight-line glide at gamma* throughout
h_dot_cal = H_DOT_MAX
z_sw_cal  = Z_GOAL / (1 + h_dot_cal*LD_MAX/V_CLIMB) * 1.2  # feasible with margin
traj_cal  = simulate(h_dot_cal, z_sw_cal, np.full(10, GAMMA_STAR))

print(f'Calibration trajectory:')
print(f'  z_sw = {traj_cal["z_sw"]:.0f} m  (sensor at z_d={Z_D:.0f} m)')
print(f'  h_sw = {traj_cal["h_sw"]:.0f} m')
print(f'  T    = {traj_cal["t"][-1]:.0f} s')


def raw_integral(traj, sensor_fn, z_d=None):
    """Integrate one sensor with kappa=1 over trajectory."""
    if z_d is None: z_d = Z_D
    n   = len(traj['t'])
    lam = np.array([sensor_fn(traj['z'][i], traj['h'][i], traj['v'][i],
                               traj['gamma'][i], traj['phase'][i], z_d)
                    for i in range(n)])
    dt_arr = np.diff(traj['t'])
    return float(np.sum(0.5*(lam[:-1]+lam[1:])*dt_arr))


# Measure raw integrals with kappa=1
KAPPA_AC=1.0; KAPPA_DOPP=1.0; KAPPA_RCS=1.0; KAPPA_CAM=1.0

I_ac   = raw_integral(traj_cal, lam_acoustic_np)
I_dopp = raw_integral(traj_cal, lam_doppler_np)
I_rcs  = raw_integral(traj_cal, lam_rcs_np)
I_cam  = raw_integral(traj_cal, lam_camera_np)

print(f'\nRaw integrals (kappa=1):')
print(f'  acoustic : {I_ac:.4e}')
print(f'  doppler  : {I_dopp:.4e}')
print(f'  rcs      : {I_rcs:.4e}')
print(f'  camera   : {I_cam:.4e}')

# Scale kappas so each sensor integral = TARGET
KAPPA_AC   = TARGET_PER_SENSOR / I_ac   if I_ac   > 0 else 1e-6
KAPPA_DOPP = TARGET_PER_SENSOR / I_dopp if I_dopp > 0 else 1e-6
KAPPA_RCS  = TARGET_PER_SENSOR / I_rcs  if I_rcs  > 0 else 1e-6
KAPPA_CAM  = TARGET_PER_SENSOR / I_cam  if I_cam  > 0 else 1e-6

print(f'\nCalibrated kappas:')
print(f'  KAPPA_AC   = {KAPPA_AC:.4e}')
print(f'  KAPPA_DOPP = {KAPPA_DOPP:.4e}')
print(f'  KAPPA_RCS  = {KAPPA_RCS:.4e}')
print(f'  KAPPA_CAM  = {KAPPA_CAM:.4e}')

# Verify
costs_cal = compute_costs(traj_cal)
print(f'\nVerification on calibration trajectory:')
print(f'  xi   = {costs_cal["xi"]:.4f}  (expected ~{TARGET_PER_SENSOR*4:.2f})')
print(f'  J_PD = {costs_cal["J_PD"]:.4f}  (target ~0.45)')

---
## Section 6 — Calibration Verification

In [ ]:
# Recompute sensor arrays for plotting
n_cal = len(traj_cal['t'])
lam_ac_arr = np.array([lam_acoustic_np(traj_cal['z'][i], traj_cal['h'][i],
                        traj_cal['v'][i], traj_cal['gamma'][i], traj_cal['phase'][i])
                        for i in range(n_cal)])
lam_dp_arr = np.array([lam_doppler_np(traj_cal['z'][i], traj_cal['h'][i],
                        traj_cal['v'][i], traj_cal['gamma'][i], traj_cal['phase'][i])
                        for i in range(n_cal)])
lam_rc_arr = np.array([lam_rcs_np(traj_cal['z'][i], traj_cal['h'][i],
                        traj_cal['v'][i], traj_cal['gamma'][i], traj_cal['phase'][i])
                        for i in range(n_cal)])
lam_cm_arr = np.array([lam_camera_np(traj_cal['z'][i], traj_cal['h'][i],
                        traj_cal['v'][i], traj_cal['gamma'][i], traj_cal['phase'][i])
                        for i in range(n_cal)])
cutoff_t_cal = traj_cal['t'][np.argmax(traj_cal['phase'] == 2)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Sensor Calibration Verification', fontsize=12)

# Trajectory
ph1 = traj_cal['phase'] == 1
ph2 = traj_cal['phase'] == 2
axes[0].plot(traj_cal['z'][ph1], traj_cal['h'][ph1], 'b', lw=2, label='Phase 1')
axes[0].plot(traj_cal['z'][ph2], traj_cal['h'][ph2], 'orange', lw=2, label='Phase 2')
axes[0].axvline(Z_D, color='red', ls='--', lw=1.5, label=f'Sensor z_d={Z_D:.0f}m')
axes[0].scatter([Z_D], [0], color='red', s=100, zorder=6, marker='v')
axes[0].set_xlabel('z [m]'); axes[0].set_ylabel('h [m]')
axes[0].set_title('Calibration trajectory')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Individual sensor rates
for arr, color, label in [(lam_ac_arr,'blue','acoustic'),
                           (lam_dp_arr,'orange','doppler'),
                           (lam_rc_arr,'purple','rcs'),
                           (lam_cm_arr,'green','camera')]:
    axes[1].plot(traj_cal['t'], arr, color=color, lw=1.5, label=label)
axes[1].axvline(cutoff_t_cal, color='gray', ls='--', lw=1, label='engine cutoff')
axes[1].set_xlabel('time [s]'); axes[1].set_ylabel('hazard rate [1/s]')
axes[1].set_title('Calibrated sensor rates')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Running J_PD
axes[2].plot(traj_cal['t'], costs_cal['jpd_run'], 'crimson', lw=2)
axes[2].axvline(cutoff_t_cal, color='gray', ls='--', lw=1, label='engine cutoff')
axes[2].set_xlabel('time [s]'); axes[2].set_ylabel('cumulative P_D')
axes[2].set_title(f'Running J_PD  final={costs_cal["J_PD"]:.4f}')
axes[2].set_ylim([0, 1]); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calibration_verification.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Initial Guess for NLP

A good initial guess is critical for IPOPT convergence.
We use a straight-line glide at `gamma*` as the warm start.
This is feasible by construction and physically reasonable.

In [ ]:
def make_initial_guess(z_d=Z_D, N=N_COLL):
    """
    Construct a feasible initial guess for the CasADi NLP.
    Uses straight-line glide at gamma* from a feasible (z_sw, h_sw).

    Returns dict with all NLP initial values.
    """
    h_dot_0  = H_DOT_MAX * 0.6
    # Minimum feasible z_sw for this h_dot
    z_sw_min = Z_GOAL / (1 + h_dot_0 * LD_MAX / V_CLIMB)
    z_sw_0   = min(z_sw_min * 1.15, z_d - 200.0)

    t_c     = z_sw_0 / V_CLIMB
    h_sw_0  = h_dot_0 * t_c
    delta_0 = z_sw_0 + h_sw_0 * LD_MAX - Z_GOAL

    assert delta_0 >= 0,   f'Initial guess infeasible: delta={delta_0:.1f}'
    assert z_sw_0 < z_d,   f'Initial guess z_sw={z_sw_0:.0f} >= z_d={z_d:.0f}'

    # Straight-line at gamma*: time bounded by both h=0 and z=Z_GOAL
    v_g      = V_STAR
    cos_g    = np.cos(abs(GAMMA_STAR))
    sin_g    = np.sin(GAMMA_STAR)           # negative
    T_h0     = h_sw_0 / (v_g * abs(sin_g)) # time to reach h=0
    T_zgoal  = (Z_GOAL - z_sw_0) / (v_g * cos_g)  # time to reach Z_GOAL
    T_glide_0 = min(T_h0, T_zgoal)

    t_nodes  = np.linspace(0, T_glide_0, N+1)
    z_nodes  = np.minimum(z_sw_0 + v_g*cos_g*t_nodes, Z_GOAL)
    h_nodes  = np.maximum(h_sw_0 + v_g*sin_g*t_nodes, 0.0)
    dr_nodes = np.maximum(np.linspace(delta_0, 0.0, N+1), 0.0)
    g_nodes  = np.full(N+1, GAMMA_STAR)

    print(f'Initial guess:')
    print(f'  h_dot   = {h_dot_0:.2f} m/s')
    print(f'  z_sw    = {z_sw_0:.0f} m  (sensor at z_d={z_d:.0f} m)')
    print(f'  h_sw    = {h_sw_0:.0f} m')
    print(f'  delta   = {delta_0:.0f} m')
    print(f'  T_glide = {T_glide_0:.0f} s')

    return {
        'h_dot':    h_dot_0,
        'z_sw':     z_sw_0,
        'h_sw':     h_sw_0,
        'delta':    delta_0,
        'T_glide':  T_glide_0,
        't_nodes':  t_nodes,
        'z_nodes':  z_nodes,
        'h_nodes':  h_nodes,
        'dr_nodes': dr_nodes,
        'g_nodes':  g_nodes,
    }


ig = make_initial_guess(Z_D, N_COLL)

---
## Section 8 — Build NLP (CasADi)

**Decision variables:**
- Scalars: `h_dot`, `z_sw`, `T_glide`
- At each collocation node k=0..N: state `x_k = [z_k, h_k, Delta_rem_k]` and control `u_k = gamma_k`
- Total: `3 + (N+1)*4` variables

**Hermite-Simpson collocation** enforces dynamics at each interval:
```
x_{k+1} = x_k + dt/6 * [f_k + 4*f_mid + f_{k+1}]
x_mid = (x_k + x_{k+1})/2 + dt/8 * [f_k - f_{k+1}]
```

**Objective (log-sum-exp):**
```
min  alpha_1 * xi(T) + alpha_2 * T_norm
```
where `xi` is accumulated via Simpson integration of the hazard rates.

In [ ]:
N = N_COLL

# ── Symbolic dynamics ─────────────────────────────────────────────────────────
# State: x = [z, h, Delta_rem]    Control: u = [gamma]
x_sym = ca.MX.sym('x', 3)
u_sym = ca.MX.sym('u', 1)

z_s      = x_sym[0]
h_s      = x_sym[1]
dr_s     = x_sym[2]
gamma_s  = u_sym[0]

v_s      = glide_speed_ca(gamma_s)
zdot_s   = v_s * ca.cos(ca.fabs(gamma_s))
hdot_s   = v_s * ca.sin(gamma_s)                  # negative

# Range budget dynamics: d(Delta_rem)/dt = -(cot(gamma*) - cot(gamma))*|dh/dt|
tan_g_s  = ca.tan(ca.fabs(gamma_s))
cot_g_s  = 1.0 / ca.fmax(tan_g_s, 1e-6)
drdot_s  = -(float(LD_MAX) - cot_g_s) * ca.fabs(hdot_s)

f_expr   = ca.vertcat(zdot_s, hdot_s, drdot_s)
L_expr   = lam_total_ca(z_s, h_s, gamma_s)        # running cost (hazard rate)

f_fn     = ca.Function('f', [x_sym, u_sym], [f_expr])
L_fn     = ca.Function('L', [x_sym, u_sym], [L_expr])

# ── NLP scalar parameters ─────────────────────────────────────────────────────
h_dot_var = ca.MX.sym('h_dot')
z_sw_var  = ca.MX.sym('z_sw')
T_glide_v = ca.MX.sym('T_glide')

# h_sw and delta derived from (h_dot, z_sw)
h_sw_var  = h_dot_var * z_sw_var / V_CLIMB
delta_0_v = z_sw_var + h_sw_var * LD_MAX - Z_GOAL
dt_v      = T_glide_v / N

# ── Collocation variables ─────────────────────────────────────────────────────
X = [ca.MX.sym(f'x{k}', 3) for k in range(N+1)]
U = [ca.MX.sym(f'u{k}', 1) for k in range(N+1)]

# ── Assemble decision variable vector and bounds ──────────────────────────────
w_list = [h_dot_var, z_sw_var, T_glide_v]
lbw    = [0.1,                   # h_dot min
          Z_START + 100,         # z_sw min (must be feasible)
          10.0]                  # T_glide min
ubw    = [H_DOT_MAX,
          Z_D - 100,             # z_sw < Z_D (engine before sensor)
          T_MAX * 3]             # generous T_glide upper bound

for k in range(N+1):
    w_list += [X[k], U[k]]
    lbw    += [Z_START, 0.0,  0.0,   GAMMA_MIN]
    ubw    += [Z_GOAL,  1e4,  1e6,   GAMMA_MAX]

w   = ca.vertcat(*w_list)
lbw = np.array(lbw)
ubw = np.array(ubw)

# ── Constraints ───────────────────────────────────────────────────────────────
g_list = []; lbg = []; ubg = []

# 1. Delta >= 0 (feasibility)
g_list.append(delta_0_v);          lbg.append(0.0);    ubg.append(1e6)

# 2. Initial state = switch point
g_list.append(X[0][0] - z_sw_var); lbg.append(0.0);    ubg.append(0.0)
g_list.append(X[0][1] - h_sw_var); lbg.append(0.0);    ubg.append(0.0)
g_list.append(X[0][2] - delta_0_v);lbg.append(0.0);    ubg.append(0.0)

# 3. Terminal: h(T) = 0
g_list.append(X[N][1]);             lbg.append(0.0);    ubg.append(0.0)

# 4. Terminal: z(T) near Z_GOAL (small tolerance for numerical reasons)
g_list.append(X[N][0] - Z_GOAL);   lbg.append(-500.0); ubg.append(500.0)

# 5. Hermite-Simpson collocation + running cost accumulation
xi_total = 0.0

for k in range(N):
    xk  = X[k];   uk  = U[k]
    xk1 = X[k+1]; uk1 = U[k+1]

    fk   = f_fn(xk,  uk)
    fk1  = f_fn(xk1, uk1)

    # Hermite midpoint interpolation
    x_mid = 0.5*(xk + xk1) + dt_v/8.0*(fk - fk1)
    u_mid = 0.5*(uk + uk1)
    f_mid = f_fn(x_mid, u_mid)

    # Collocation defect (must = 0)
    defect = xk1 - xk - dt_v/6.0*(fk + 4*f_mid + fk1)
    g_list.append(defect); lbg += [0.0]*3; ubg += [0.0]*3

    # Running cost (hazard integral via Simpson)
    Lk   = L_fn(xk,    uk)
    Lk1  = L_fn(xk1,   uk1)
    Lmid = L_fn(x_mid, u_mid)
    xi_total += dt_v/6.0 * (Lk + 4*Lmid + Lk1)

    # Path constraints: Delta_rem >= 0, h >= 0
    g_list.append(X[k][2]); lbg.append(0.0); ubg.append(1e6)
    g_list.append(X[k][1]); lbg.append(0.0); ubg.append(1e4)

# ── Objective ─────────────────────────────────────────────────────────────────
# Phase 1 time (deterministic given z_sw and v_climb)
T_phase1  = z_sw_var / V_CLIMB
T_total_v = T_phase1 + T_glide_v
T_norm_v  = T_total_v / T_MAX

# Log-sum-exp: minimize xi directly (equiv. to minimizing J_PD)
obj = ALPHA_1 * xi_total + ALPHA_2 * T_norm_v

# ── Build NLP ─────────────────────────────────────────────────────────────────
g   = ca.vertcat(*g_list)
nlp = {'x': w, 'f': obj, 'g': g}

opts = {
    'ipopt.print_level':    5,
    'ipopt.max_iter':       1000,
    'ipopt.tol':            1e-6,
    'ipopt.acceptable_tol': 1e-4,
    'print_time':           True,
}
solver = ca.nlpsol('solver', 'ipopt', nlp, opts)

n_vars = w.shape[0]
n_cons = g.shape[0]
print(f'NLP built:')
print(f'  Decision variables: {n_vars}')
print(f'  Constraints:        {n_cons}')

---
## Section 9 — Solve with IPOPT

In [ ]:
# Assemble initial guess vector
w0 = [ig['h_dot'], ig['z_sw'], ig['T_glide']]
for k in range(N+1):
    w0 += [ig['z_nodes'][k], ig['h_nodes'][k], ig['dr_nodes'][k],
           ig['g_nodes'][k]]
w0 = np.array(w0)

print(f'Solving NLP with IPOPT...')
print(f'Initial guess: h_dot={ig["h_dot"]:.2f}, z_sw={ig["z_sw"]:.0f}m, '
      f'h_sw={ig["h_sw"]:.0f}m')

sol    = solver(x0=w0, lbx=lbw, ubx=ubw, lbg=np.array(lbg), ubg=np.array(ubg))
w_opt  = sol['x'].full().flatten()
J_opt  = float(sol['f'])
stats  = solver.stats()

print(f'\nIPOPT status: {stats["return_status"]}')
print(f'Objective (NLP): {J_opt:.6f}')

---
## Section 10 — Extract and Verify Solution

In [ ]:
# ── Extract solution ─────────────────────────────────────────────────────────
h_dot_opt = float(w_opt[0])
z_sw_opt  = float(w_opt[1])
T_glide   = float(w_opt[2])
h_sw_opt  = h_dot_opt * z_sw_opt / V_CLIMB
delta_opt = z_sw_opt + h_sw_opt * LD_MAX - Z_GOAL

off   = 3
z_tr  = np.array([w_opt[off + k*4 + 0] for k in range(N+1)])
h_tr  = np.array([w_opt[off + k*4 + 1] for k in range(N+1)])
dr_tr = np.array([w_opt[off + k*4 + 2] for k in range(N+1)])
g_tr  = np.array([w_opt[off + k*4 + 3] for k in range(N+1)])
t_tr  = np.linspace(0, T_glide, N+1)
v_tr  = np.array([glide_speed_np(g) for g in g_tr])

# Evaluate costs numerically on extracted trajectory
lam_tr  = np.array([lam_total_np(z_tr[k], h_tr[k], v_tr[k], g_tr[k], 2)
                    for k in range(N+1)])
dt_tr   = np.diff(t_tr)
xi_opt  = float(np.sum(0.5*(lam_tr[:-1]+lam_tr[1:])*dt_tr))
J_PD_opt = 1.0 - np.exp(-xi_opt)
T_total_opt = z_sw_opt/V_CLIMB + T_glide
T_norm_opt  = T_total_opt / T_MAX
J_A_opt = ALPHA_1 * xi_opt + ALPHA_2 * T_norm_opt

# Running J_PD
cumint_tr = np.concatenate([[0], np.cumsum(0.5*(lam_tr[:-1]+lam_tr[1:])*dt_tr)])
jpd_run_tr = 1.0 - np.exp(-cumint_tr)

print('Optimal attacker decisions:')
print(f'  h_dot   = {h_dot_opt:.3f} m/s')
print(f'  z_sw    = {z_sw_opt:.1f} m  (engine cuts BEFORE sensor at z_d={Z_D:.0f} m)')
print(f'  h_sw    = {h_sw_opt:.1f} m')
print(f'  delta   = {delta_opt:.1f} m  (range budget)')
print(f'  T_glide = {T_glide:.1f} s')
print(f'  T_total = {T_total_opt:.1f} s')
print(f'\nCosts:')
print(f'  xi(T)   = {xi_opt:.4f}  (hazard integral)')
print(f'  J_PD    = {J_PD_opt:.4f}  (= 1 - exp(-xi))')
print(f'  T_norm  = {T_norm_opt:.4f}')
print(f'  J_A     = {J_A_opt:.4f}  (NLP obj = {J_opt:.4f})')
print(f'\nGamma profile:')
print(f'  min  = {np.degrees(g_tr.min()):.2f} deg')
print(f'  max  = {np.degrees(g_tr.max()):.2f} deg')
print(f'  mean = {np.degrees(g_tr.mean()):.2f} deg')
print(f'  std  = {np.degrees(g_tr.std()):.4f} deg')
print(f'  gamma* = {np.degrees(GAMMA_STAR):.2f} deg  (best glide)')

---
## Section 11 — Comparison: Optimal vs Straight-Line Gamma*

Compare the CasADi optimal trajectory against a naive straight-line
glide at `gamma*` using the same `(h_dot, z_sw)`.
If curved beats straight, the Bellman optimal control is non-trivial.

In [ ]:
# Straight-line at gamma* with same (h_dot, z_sw)
N_SEG_COMP = 10
traj_straight = simulate(h_dot_opt, z_sw_opt, np.full(N_SEG_COMP, GAMMA_STAR))
costs_straight = compute_costs(traj_straight)

# Straight-line at gamma* with same (h_dot, z_sw) but varying gamma from NLP
# (for a fair comparison: segment the NLP gamma profile)
g_segments = np.array([np.mean(g_tr[int(k*N/N_SEG_COMP):int((k+1)*N/N_SEG_COMP)])
                        for k in range(N_SEG_COMP)])
traj_segmented = simulate(h_dot_opt, z_sw_opt, g_segments)
costs_segmented = compute_costs(traj_segmented)

print(f'Comparison at z_d = {Z_D:.0f} m:')
print(f'{"Strategy":<25} {"J_A":>8} {"J_PD":>8} {"xi":>8} {"T_norm":>8}')
print('-' * 60)
print(f'{"CasADi optimal":<25} {J_A_opt:>8.4f} {J_PD_opt:>8.4f} '
      f'{xi_opt:>8.4f} {T_norm_opt:>8.4f}')
print(f'{"Straight gamma*":<25} {costs_straight["J_A"]:>8.4f} '
      f'{costs_straight["J_PD"]:>8.4f} {costs_straight["xi"]:>8.4f} '
      f'{costs_straight["T_norm"]:>8.4f}')
print(f'{"Segmented (NLP gamma)":<25} {costs_segmented["J_A"]:>8.4f} '
      f'{costs_segmented["J_PD"]:>8.4f} {costs_segmented["xi"]:>8.4f} '
      f'{costs_segmented["T_norm"]:>8.4f}')

improvement = costs_straight['J_A'] - J_A_opt
print(f'\nJ_A improvement (optimal vs straight): {improvement:.4f}')
if improvement > 1e-4:
    print('CURVED TRAJECTORY OUTPERFORMS STRAIGHT LINE')
    print('Bellman optimal control is non-trivial for this sensor geometry.')
else:
    print('No significant improvement — sensors may need recalibration or')
    print('the optimal straight-line already minimizes this sensor geometry.')

---
## Section 12 — Results Plots

In [ ]:
# Reconstruct powered phase for plotting
dt_p = 0.5
z_p, h_p, t_p = [Z_START], [0.0], [0.0]
t_, z_, h_ = 0.0, Z_START, 0.0
gamma_p = np.arctan2(h_dot_opt, V_CLIMB)
while h_ < h_sw_opt - 1e-3 and z_ < z_sw_opt - 1e-3:
    z_ += V_CLIMB*dt_p; h_ += h_dot_opt*dt_p; t_ += dt_p
    z_p.append(z_); h_p.append(h_); t_p.append(t_)
z_p = np.array(z_p); h_p = np.array(h_p); t_p = np.array(t_p)
t_abs = t_tr + t_p[-1]   # absolute time for glide phase

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    f'A1b: Bellman Optimal Attacker Trajectory (CasADi + IPOPT)\n'
    f'z_d={Z_D:.0f}m,  J_A={J_A_opt:.4f},  J_PD={J_PD_opt:.4f},  '
    f'N_coll={N_COLL},  status={stats["return_status"]}',
    fontsize=11
)

# ── Plot 1: Trajectory shape ─────────────────────────────────────────────────
axes[0,0].plot(z_p, h_p, color='royalblue', lw=2.5, label='Phase 1 (powered)')
axes[0,0].plot(z_tr, h_tr, color='darkorange', lw=2.5, label='Phase 2 (optimal)')

# Straight-line gamma* for comparison
z_st_plot = np.linspace(z_sw_opt, Z_GOAL, 200)
h_st_plot = np.maximum(h_sw_opt + (z_st_plot - z_sw_opt)*np.tan(GAMMA_STAR), 0)
axes[0,0].plot(z_st_plot, h_st_plot, 'g--', lw=1.5,
               label=f'Straight gamma*={np.degrees(GAMMA_STAR):.1f}°')

axes[0,0].axvline(Z_D, color='red', ls='--', lw=1.5, label=f'Sensor z_d={Z_D:.0f}m')
axes[0,0].scatter([Z_D], [0], color='red', s=120, zorder=6, marker='v')
axes[0,0].scatter([z_sw_opt], [h_sw_opt], color='royalblue', s=100,
                  zorder=6, marker='x', lw=2.5,
                  label=f'Engine cutoff ({z_sw_opt:.0f}m, {h_sw_opt:.0f}m)')
axes[0,0].set_xlabel('z [m]'); axes[0,0].set_ylabel('h [m]')
axes[0,0].set_title('Trajectory (key: curved vs straight-line)')
axes[0,0].legend(fontsize=7); axes[0,0].grid(True, alpha=0.3)

# ── Plot 2: Gamma vs time ─────────────────────────────────────────────────────
axes[0,1].plot(t_tr, np.degrees(g_tr), color='darkorange', lw=2)
axes[0,1].axhline(np.degrees(GAMMA_STAR), color='green', ls='--', lw=1.5,
                   label=f'gamma*={np.degrees(GAMMA_STAR):.1f}°')
axes[0,1].set_xlabel('time [s] (glide phase only)')
axes[0,1].set_ylabel('gamma [deg]')
axes[0,1].set_title('Optimal gamma(t)\n(curved = Bellman solution active)')
axes[0,1].legend(fontsize=9); axes[0,1].grid(True, alpha=0.3)

# ── Plot 3: Gamma vs altitude ─────────────────────────────────────────────────
axes[0,2].plot(np.degrees(g_tr), h_tr, color='purple', lw=2)
axes[0,2].axvline(np.degrees(GAMMA_STAR), color='green', ls='--', lw=1.5,
                   label=f'gamma*')
axes[0,2].set_xlabel('gamma [deg]'); axes[0,2].set_ylabel('h [m]')
axes[0,2].set_title('Gamma vs altitude\n(shows how strategy changes with altitude)')
axes[0,2].legend(fontsize=9); axes[0,2].grid(True, alpha=0.3)

# ── Plot 4: Trim airspeed ─────────────────────────────────────────────────────
axes[1,0].plot(t_tr, v_tr, color='black', lw=2, label='optimal v(gamma(t))')
axes[1,0].axhline(V_STAR, color='green', ls='--', lw=1.5,
                   label=f'v*={V_STAR:.1f} m/s')
axes[1,0].set_xlabel('time [s] (glide phase)')
axes[1,0].set_ylabel('v [m/s]')
axes[1,0].set_title('Trim airspeed v(gamma(t))\n(nonholonomic: v determined by gamma)')
axes[1,0].legend(fontsize=9); axes[1,0].grid(True, alpha=0.3)

# ── Plot 5: Hazard rate ───────────────────────────────────────────────────────
axes[1,1].plot(t_tr, lam_tr, color='crimson', lw=2, label='optimal')
# Compute for straight-line comparison
lam_str = np.array([lam_total_np(traj_straight['z'][i], traj_straight['h'][i],
                                  traj_straight['v'][i], traj_straight['gamma'][i], 2)
                    for i in range(len(traj_straight['t']))
                    if traj_straight['phase'][i] == 2])
t_str = traj_straight['t'][traj_straight['phase'] == 2] -         traj_straight['t'][np.argmax(traj_straight['phase']==2)]
axes[1,1].plot(t_str, lam_str, 'g--', lw=1.5, label='straight gamma*')
axes[1,1].set_xlabel('time [s] (glide phase)')
axes[1,1].set_ylabel('total hazard rate [1/s]')
axes[1,1].set_title('Hazard rate comparison')
axes[1,1].legend(fontsize=9); axes[1,1].grid(True, alpha=0.3)

# ── Plot 6: Running J_PD ─────────────────────────────────────────────────────
axes[1,2].plot(t_abs, jpd_run_tr, color='crimson', lw=2,
               label=f'optimal  J_PD={J_PD_opt:.4f}')
axes[1,2].plot(traj_straight['t'][traj_straight['phase']==2] +
               traj_straight['t'][np.argmax(traj_straight['phase']==1)],
               costs_straight['jpd_run'][traj_straight['phase']==2],
               'g--', lw=1.5, label=f'straight  J_PD={costs_straight["J_PD"]:.4f}')
axes[1,2].set_xlabel('time [s]')
axes[1,2].set_ylabel('cumulative P_D')
axes[1,2].set_title(f'Running J_PD(t)')
axes[1,2].set_ylim([0, 1])
axes[1,2].legend(fontsize=9); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('a1b_bellman_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: a1b_bellman_result.png')

---
## Section 13 — Summary

In [ ]:
print('=' * 60)
print('A1b BELLMAN / DIRECT COLLOCATION RESULTS')
print('=' * 60)
print(f'Corridor:  z=[{Z_START:.0f}, {Z_GOAL:.0f}] m')
print(f'Sensor:    z_d={Z_D:.0f} m  ({100*Z_D/Z_GOAL:.0f}% of corridor)')
print(f'Method:    Hermite-Simpson, N={N_COLL} intervals, IPOPT')
print(f'Status:    {stats["return_status"]}')
print(f'\nOptimal Phase 1 decisions:')
print(f'  h_dot = {h_dot_opt:.3f} m/s  (climb rate)')
print(f'  z_sw  = {z_sw_opt:.1f} m    (engine cutoff position)')
print(f'  h_sw  = {h_sw_opt:.1f} m    (engine cutoff altitude)')
print(f'  delta = {delta_opt:.1f} m   (range budget)')
print(f'\nOptimal Phase 2 gamma profile:')
print(f'  gamma(t) min = {np.degrees(g_tr.min()):.2f} deg')
print(f'  gamma(t) max = {np.degrees(g_tr.max()):.2f} deg')
print(f'  gamma(t) std = {np.degrees(g_tr.std()):.4f} deg')
print(f'  gamma*       = {np.degrees(GAMMA_STAR):.2f} deg  (best glide reference)')
print(f'\nCosts at equilibrium:')
print(f'  xi(T)  = {xi_opt:.4f}  (total hazard integral)')
print(f'  J_PD   = {J_PD_opt:.4f}  (detection probability)')
print(f'  T_norm = {T_norm_opt:.4f}')
print(f'  J_A    = {J_A_opt:.4f}')
print(f'\nComparison vs straight-line gamma*:')
print(f'  J_A improvement = {costs_straight["J_A"] - J_A_opt:.4f}')
g_std = np.degrees(g_tr.std())
if g_std > 2.0:
    print(f'  Gamma std={g_std:.2f} deg > 2 deg: CURVED trajectory confirmed')
    print('  Bellman optimal control is non-trivial for this geometry.')
elif g_std > 0.5:
    print(f'  Gamma std={g_std:.2f} deg: MODERATE variation')
else:
    print(f'  Gamma std={g_std:.2f} deg: nearly constant')
    print('  Adjust TARGET_PER_SENSOR or Z_D to activate three-way contradiction.')
print('=' * 60)